In [9]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "src" / "database.py").is_file():
    raise FileNotFoundError(f"Não encontrei src/database.py em {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.database import get_connection

print("Importação OK:", project_root)

Importação OK: e:\VSCODE Files\Projects\03_brazilian_fin_corp_analytics


In [10]:
from datetime import date

import pandas as pd

from src.bcb import fetch_sgs_series

series_id = 433
records = fetch_sgs_series(
    series_id=series_id,
    start_date=date(2000, 1, 1),
    end_date=date.today(),
)

records[:3]

[{'data': '01/01/2000', 'valor': '0.62'},
 {'data': '01/02/2000', 'valor': '0.13'},
 {'data': '01/03/2000', 'valor': '0.22'}]

In [11]:
df_raw = pd.DataFrame(records)
df_raw.info()
df_raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   data    322 non-null    str  
 1   valor   322 non-null    str  
dtypes: str(2)
memory usage: 5.2 KB


,data,valor
0,01/01/2000,0.62
1,01/02/2000,0.13
2,01/03/2000,0.22
3,01/04/2000,0.42
4,01/05/2000,0.01


In [12]:
reference_month = pd.to_datetime(
    df_raw["data"], format="%d/%m/%Y"
).dt.to_period("M")

observed_months = pd.PeriodIndex(reference_month.dropna().unique()).sort_values()
expected_months = pd.period_range("2000-01", observed_months.max(), freq="M")
missing_months = expected_months.difference(observed_months)

pd.Series({
    "Total de linhas": len(df_raw),
    "Primeiro mês": observed_months.min(),
    "Último mês": observed_months.max(),
    "Datas ausentes": reference_month.isna().sum(),
    "Linhas excedentes no mesmo mês": reference_month.dropna().duplicated().sum(),
    "Meses ausentes desde jan/2000": missing_months.tolist(),
    "Meses distintos em 2024": reference_month[reference_month.dt.year == 2024].nunique(),
})

Total de linhas                       322
Primeiro mês                      2000-01
Último mês                        2026-10
Datas ausentes                          0
Linhas excedentes no mesmo mês          0
Meses ausentes desde jan/2000          []
Meses distintos em 2024                12
dtype: object

In [13]:
value_numeric = pd.to_numeric(df_raw["valor"], errors="coerce")

pd.Series({
    "Valores ausentes na origem": df_raw["valor"].isna().sum(),
    "Falhas de conversão": (df_raw["valor"].notna() & value_numeric.isna()).sum(),
    "Menor variação mensal (%)": value_numeric.min(),
    "Maior variação mensal (%)": value_numeric.max(),
    "Meses com variação zero": value_numeric.eq(0).sum(),
    "Meses com variação negativa": value_numeric.lt(0).sum(),
}).to_frame("resultado")

,resultado
Valores ausentes na origem,0.00
Falhas de conversão,0.00
Menor variação mensal (%),-0.68
Maior variação mensal (%),3.02
Meses com variação zero,1.00
Meses com variação negativa,16.00


## IPCA data quality findings

- The extraction contains 322 monthly observations from January 2000 through
  October 2026, with one observation per month and no gaps or duplicates.
- All 322 reported values were present and converted successfully to numeric form.
- Monthly IPCA ranged from −0.68% to 3.02%. Inflation was negative in 16 months,
  zero in 1 month, and positive in 305 months.
- For future loads, validate monthly continuity, uniqueness by series and
  reference month, and numeric conversion. Preserve zero and negative rates.
  The observed minimum and maximum describe this extraction only.

In [14]:
from src.bcb import prepare_sgs_rows

rows = prepare_sgs_rows(series_id, records)
rows[:3]

[(433, datetime.date(2000, 1, 1), '0.62'),
 (433, datetime.date(2000, 2, 1), '0.13'),
 (433, datetime.date(2000, 3, 1), '0.22')]

In [15]:
from src.bcb import load_sgs_observations

affected_rows = load_sgs_observations(rows)
print(f"Linhas inseridas ou atualizadas: {affected_rows}")

Linhas inseridas ou atualizadas: 0
